# LLM Fine-tuning + Eval — Resume → JSON (Colab / Kaggle T4)

Runs the full project on a **free T4**: install → build data → QLoRA fine-tune → base-vs-fine-tuned eval → view the comparison table.

**Before running:** set `Runtime → Change runtime type → GPU (T4)`.

The base model (`Qwen/Qwen2.5-3B-Instruct`) is open — no HF login or API key needed.

## 1. Get the code
Clone your repo, or upload the project folder. Edit `GITHUB_URL` to your fork.

In [3]:
import os

GITHUB_URL = "https://github.com/zeelshah1805/llm-finetune.git"  # <-- edit me
REPO_DIR = "/content/llm-finetune"

if not os.path.isdir(REPO_DIR):
    !git clone $GITHUB_URL $REPO_DIR
%cd $REPO_DIR
!ls

/content/llm-finetune
data  merge_and_push.py  PLAN.md    requirements.txt  schema.py
eval  notebooks		 README.md  results	      train.py


## 2. Install dependencies
`bitsandbytes` is the 4-bit quantization backend (CUDA only).

In [ ]:
!pip install -q -r requirements.txt
# bitsandbytes must match the runtime's CUDA/Triton; force latest to avoid
# "No module named 'triton.ops'" / missing cudaXXX.so on current Colab.
!pip install -q -U bitsandbytes
import torch, bitsandbytes
print('CUDA available:', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
print('bitsandbytes', bitsandbytes.__version__)

## 3. Build the dataset (deterministic)
The splits are already committed; this just confirms they regenerate identically. The **test split is frozen**.

In [5]:
!python data/build_dataset.py --n 400 --seed 42

wrote  320 -> /content/llm-finetune/data/train.jsonl
wrote   40 -> /content/llm-finetune/data/val.jsonl
wrote   40 -> /content/llm-finetune/data/test.jsonl

Example resume:
------------------------------------------------------------
Diya Brown | diya.163@hotmail.com | 11y exp | Skills: Airflow,Spark,Hugging Face,Redis,GCP,Kubernetes,Azure | Full Stack Developer@Razorpay(1y) Data Scientist@Helix Systems(2.5y) DevOps Engineer@Wipro(2.5y) Machine Learning Engineer@Zomato(5y) | B.Tech in Electronics-Anna University-2023
------------------------------------------------------------
Target JSON:
{
  "name": "Diya Brown",
  "email": "diya.163@hotmail.com",
  "phone": null,
  "total_years_experience": 11,
  "skills": [
    "Airflow",
    "Spark",
    "Hugging Face",
    "Redis",
    "GCP",
    "Kubernetes",
    "Azure"
  ],
  "education": [
    {
      "degree": "B.Tech in Electronics",
      "institution": "Anna University",
      "year": 2023
    }
  ],
  "work_experience": [
    {
      "ti

## 4. QLoRA fine-tune
4-bit base + LoRA adapter. ~10–20 min on a T4 for 2 epochs. Watch the train/eval loss gap for overfitting.
Saves the adapter to `adapters/qwen-resume/` and a loss curve to `assets/loss_curve.png`.

In [ ]:
!python train.py --base Qwen/Qwen2.5-3B-Instruct --epochs 2

In [ ]:
from IPython.display import Image
Image('assets/loss_curve.png')

## 5. Evaluate base vs. fine-tuned
Generates predictions for BOTH models on the same frozen test set, scores them, and writes `results/comparison.md` + charts.
Raw predictions are saved to `results/raw/` so you can re-score offline later without a GPU.

In [ ]:
!python eval/run_eval.py --base Qwen/Qwen2.5-3B-Instruct --adapter adapters/qwen-resume

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/comparison.md').read()))
display(Image('results/charts/headline.png'))
display(Image('results/charts/per_field_f1.png'))

## 6. (Optional) Merge + push to the Hugging Face Hub
Run `huggingface-cli login` first, then merge the adapter into fp16 weights and push with a model card.

In [ ]:
# from huggingface_hub import notebook_login; notebook_login()
# !python merge_and_push.py --adapter adapters/qwen-resume --push <your-username>/qwen2.5-3b-resume-json

## 7. Download artifacts
Pull the results back to your machine to commit them to the repo (the comparison table + charts are your strongest interview artifact).

In [ ]:
import shutil
shutil.make_archive('/content/artifacts', 'zip', '.', 'results')
shutil.make_archive('/content/adapter', 'zip', 'adapters', 'qwen-resume')
try:
    from google.colab import files
    files.download('/content/artifacts.zip')
    files.download('/content/adapter.zip')
except Exception as e:
    print('not on Colab or download unavailable:', e)